In [ ]:
from __future__ import annotations

import json
import pathlib
import random
from collections import Counter
from pathlib import Path

from kebab.utils.dataset.wikidata import wikidata_utils
from kebab.utils.dataset.wikidata.wikidata_utils import ResolvedWikidataEntity

In [ ]:
required_entity_type = "Q5"
linking_pair_count_limit = 10000
linking_entity_count_limit = 10
linking_property_pattern_count_limit = 50

clustering_entity_count_limit = 100
clustering_fragment_count_limit = 10

In [ ]:
# pairs
base_linking_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_dataset.jsonl"
)

# pairs ground truth
base_linking_ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Linking Dataset Base"
    / "rebel_linking_ground_truth.jsonl"
)

# clustering dataset path
base_clustering_dataset_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_clustering_dataset.jsonl"
)

# clustering dataset ground truth
base_clustering_ground_truth_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_clustering_ground_truth.jsonl"
)

# fragment_id to entity_id map
fragment_to_entity_map_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "REBEL"
    / "Clustering Dataset Base"
    / "rebel_fragment_to_entity_map.jsonl"
)

# Wikidata type hierarchy for filtering by type
type_hierarchy_path = (
    pathlib.Path.home()
    / "OneDrive - Microsoft"
    / "Benchmark"
    / "Datasets"
    / "Wikidata"
    / "Type Hierarchy"
    / "2025-01-30"
    / "wikidata_type_hierarchy.jsonl"
)

In [ ]:
# get the target entity type and its descendants
get_descendants = False

graph, type_id_to_node = wikidata_utils.load_type_hierarchy(type_hierarchy_path)
required_entity_types = (
    wikidata_utils.collect_all_subtypes(graph, type_id_to_node, required_entity_type)
    if get_descendants
    else {required_entity_type}
)
print(f"Target entity type: {required_entity_type} ({len(required_entity_types):,d} with descendants)")

In [ ]:
required_entity_types = {type_id_to_node[t]["name"] for t in required_entity_types}

# Linking
---

In [ ]:
# load the ground truth
with open(base_linking_ground_truth_path, encoding="utf-8") as f:
    ground_truth = [json.loads(line) for line in f]

# load fragment to entity map
fragment_to_entity_map = {}
with open(fragment_to_entity_map_path, encoding="utf-8") as f:
    for line in f:
        f_id, e_id = json.loads(line)
        fragment_to_entity_map[f_id] = e_id

In [ ]:
# load and filter fragments to only include the target entity types
fragments = {}
pairs = []
labels = []


def add_fragment(fragment: ResolvedWikidataEntity) -> ResolvedWikidataEntity:
    """Add the fragment to the set of known fragments."""
    if fragment.metadata["fragment_id"] not in fragments:
        fragments[fragment.metadata["fragment_id"]] = fragment
        fragment.evidence_map = None
        fragment.source_ids = None

        fragment.entity_id = fragment_to_entity_map[fragment.metadata["fragment_id"]]

    return fragments[fragment.metadata["fragment_id"]]


with open(base_linking_dataset_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        d = json.loads(line)
        left = add_fragment(ResolvedWikidataEntity.from_dict(d[0]))
        right = add_fragment(ResolvedWikidataEntity.from_dict(d[1]))

        if not set(left.wikidata_type).intersection(required_entity_types) or not set(right.wikidata_type).intersection(
            required_entity_types
        ):
            continue

        pairs.append((left, right))
        labels.append(ground_truth[i])

assert len(pairs) == len(labels)

print(f"Filtered to {len(pairs):,d} pairs where both entities are of the target types")
print(f"Positive pairs: {sum(labels):,d} ({sum(labels) / len(labels):.2%})")

In [ ]:
# sample pairs
sampled_pairs = []
sampled_labels = []

entity_counter = Counter()
property_pattern_counter = Counter()

indices = list(range(len(pairs)))
random.shuffle(indices)
iterations = 0

for i in indices:
    iterations += 1

    if len(sampled_pairs) >= linking_pair_count_limit:
        break

    pair = pairs[i]
    left, right = pair

    if (
        entity_counter[left.entity_id] >= linking_entity_count_limit
        or entity_counter[right.entity_id] >= linking_entity_count_limit
    ):
        continue

    prop_pattern = tuple(sorted([tuple(sorted(left.properties)), tuple(sorted(right.properties))]))
    if property_pattern_counter[prop_pattern] >= linking_property_pattern_count_limit:
        continue

    entity_counter[left.entity_id] += 1
    if left.entity_id != right.entity_id:
        entity_counter[right.entity_id] += 1

    property_pattern_counter[prop_pattern] += 1

    sampled_pairs.append(pairs[i])
    sampled_labels.append(labels[i])

print(f"Sampled {len(sampled_pairs):,d} pairs containing {len(entity_counter):,d} distinct entities")
print(f"Positive pairs: {sum(sampled_labels):,d} ({sum(sampled_labels) / len(sampled_labels):.2%})")
print(f"Iterations: {iterations:,d}, acceptance rate {len(sampled_pairs) / iterations:.2%}")

# top entities
print("\nTop entities:")
for i, (k, v) in enumerate(entity_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")

# top property patterns
print("\nTop property patterns:")
for i, (k, v) in enumerate(property_pattern_counter.most_common(n=100)):
    print(f"{i}: {k}: {v}")

In [ ]:
# write the sampled pairs and labels to files
sampled_dataset_path = Path.cwd() / "sampled_rebel_linking_dataset.jsonl"

with open(sampled_dataset_path, "w", encoding="utf-8") as f:
    for pair in sampled_pairs:
        d = [
            pair[0].to_dict(minimal_repr=True),
            pair[1].to_dict(minimal_repr=True),
        ]
        f.write(json.dumps(d) + "\n")

sampled_ground_truth_path = Path.cwd() / "sampled_rebel_linking_ground_truth.jsonl"

with open(sampled_ground_truth_path, "w", encoding="utf-8") as f:
    for label in sampled_labels:
        f.write(json.dumps(label) + "\n")

# Clustering
---

In [ ]:
# filter the clustering dataset and ground truth to only include the sampled entities
allowed_entity_ids = set(entity_counter)
entity_counter = Counter()

with (
    open(base_clustering_dataset_path, encoding="utf-8") as f_ds,
    open(base_clustering_ground_truth_path, encoding="utf-8") as f_gt,
    open("sampled_rebel_clustering_dataset.jsonl", "w", encoding="utf-8") as f_ds_out,
    open("sampled_rebel_clustering_ground_truth.jsonl", "w", encoding="utf-8") as f_gt_out,
):
    for line_ds, line_gt in zip(f_ds, f_gt, strict=False):
        fragment = json.loads(line_ds)
        entity_id = json.loads(line_gt)

        if entity_id not in allowed_entity_ids:
            continue

        if entity_id not in entity_counter and len(entity_counter) == clustering_entity_count_limit:
            continue

        if entity_counter[entity_id] >= clustering_fragment_count_limit:
            continue

        entity_counter[entity_id] += 1

        f_ds_out.write(line_ds)
        f_gt_out.write(line_gt)

print(
    f"Filtered to {sum(entity_counter.values()):,d} fragments for {len(entity_counter):,d} entities, allowed = {len(allowed_entity_ids):,d} entities, target = {clustering_entity_count_limit:,d} entities"
)

print("\nTop entities:")
for i, (k, v) in enumerate(entity_counter.most_common(n=10)):
    print(f"{i}: {k}: {v}")